In [0]:
import os, sys, json
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src import config_loader as cl

cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"))
log = cl.meta_table(cfg, "run_log")
wm = cl.meta_table(cfg, "watermark")
print("ready | catalog:", cfg["project"]["catalog"])

In [0]:
# Which runs exist? Find the first incremental run's run_id
display(spark.sql(f"""
  SELECT run_id, pipeline_name, min(started_at) AS first_activity, count(*) AS log_rows
  FROM {log} GROUP BY run_id, pipeline_name ORDER BY first_activity
"""))

In [0]:
# Paste the run_id of your first incremental run (pipeline_name = manual_incremental, earliest one)
RUN_ID = "20261127T073840_fb7cf535"

display(spark.sql(f"""
  SELECT stage, resource_type, load_date, status, pages_fetched, records_read,
         records_written, truncated, started_at, ended_at, data_saved_at
  FROM {log} WHERE run_id = '{RUN_ID}'
  ORDER BY started_at
"""))

In [0]:
# The watermark's own change history: one MERGE per resource per day loaded
display(spark.sql(f"DESCRIBE HISTORY {wm}").select("version", "timestamp", "operation", "operationMetrics"))

In [0]:
t = cl.resource_table(cfg, "bronze", "Observation")
display(spark.sql(f"""
  SELECT coalesce(nullif(regexp_extract(subj, '^(?:.*/)?([A-Za-z]+)/', 1), ''),
                  CASE WHEN subj IS NULL THEN '(no subject)' ELSE '(other format)' END) AS subject_type,
         count(*) AS n
  FROM (SELECT get_json_object(resource_json, '$.subject.reference') AS subj FROM {t})
  GROUP BY 1 ORDER BY n DESC
"""))

In [0]:

state = json.load(open(os.path.join(os.path.dirname(cl.raw_root(cfg)), "_demo", "scd2_demo_patients.json")))
ids = "','".join([state["alpha"]["id"], state["bravo"]["id"]])
t = cl.resource_table(cfg, "silver", "Patient")
display(spark.sql(f"""
  SELECT patient_id, family_name, version_number, is_current, effective_from, effective_to,
         city, marital_status_code, meta_version_id, extraction_timestamp, load_date
  FROM {t} WHERE patient_id IN ('{ids}') ORDER BY patient_id, version_number
"""))

In [0]:
import json
state = json.load(open(os.path.join(os.path.dirname(cl.raw_root(cfg)), "_demo", "scd2_demo_patients.json")))
bravo_id = state["bravo"]["id"]
t = cl.resource_table(cfg, "silver", "Patient")
display(spark.sql(f"""
  SELECT patient_id, full_name, version_number, is_current, effective_from, effective_to,
         city, marital_status_code, meta_version_id, load_date
  FROM {t}
  WHERE patient_id IN (SELECT patient_id FROM {t} GROUP BY patient_id HAVING count(*) > 1)
     OR patient_id = '{bravo_id}'
  ORDER BY patient_id, version_number
"""))

In [0]:
gold_map = {"Patient": "dim_patient", "Encounter": "fact_encounter",
            "Observation": "fact_observation", "Condition": "fact_condition"}
GOLD = f"{cfg['project']['catalog']}.{cfg['storage']['layers']['gold']}"
rows = []
for r in cl.get_resources(cfg):
    n = r["name"]
    silver_t = cl.resource_table(cfg, "silver", n)
    rows.append({
        "resource": n,
        "bronze_rows": spark.table(cl.resource_table(cfg, "bronze", n)).count(),
        "silver_rows_all_versions": spark.table(silver_t).count(),
        "silver_current_rows": spark.sql(f"SELECT count(*) FROM {silver_t} WHERE is_current").collect()[0][0],
        "gold_rows": spark.table(f"{GOLD}.{gold_map[n]}").count()})
display(pd.DataFrame(rows))

In [0]:
log = cl.meta_table(cfg, "run_log")
display(spark.sql(f"""
  SELECT run_id, pipeline_name, stage, resource_type, load_date, status,
         records_read, records_written, started_at, ended_at, data_saved_at
  FROM {log} ORDER BY logged_at DESC LIMIT 40"""))

In [0]:
display(spark.sql(f"""
  SELECT run_id, pipeline_name, stage,
         count(*) AS log_rows,
         sum(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS succeeded,
         sum(CASE WHEN status <> 'SUCCESS' THEN 1 ELSE 0 END) AS failed,
         sum(records_written) AS records_written,
         min(started_at) AS first_start, max(ended_at) AS last_end
  FROM {log}
  WHERE run_id = '567432434734687'
  GROUP BY run_id, pipeline_name, stage
  ORDER BY first_start
"""))

In [0]:
b = cl.resource_table(cfg, "bronze", "Observation")
s = cl.resource_table(cfg, "silver", "Observation")
display(spark.sql(f"""
  SELECT b.resource_id, b.meta_version_id, b.meta_last_updated, b.load_date, b.extraction_timestamp
  FROM {b} b
  LEFT ANTI JOIN {s} s
    ON b.resource_id = s.observation_id AND b.meta_version_id <=> s.meta_version_id
"""))

In [0]:
from src import silver as sv
rid = "886361"

display(spark.sql(f"""SELECT observation_id, version_number, is_current, effective_from,
                             meta_version_id, status, code, value_number, row_hash
                      FROM {s} WHERE observation_id = '{rid}' ORDER BY version_number"""))

df = sv.build_silver_df(spark, cfg, "Observation")
display(df.where(f"observation_id = '{rid}'")
          .select("observation_id", "meta_version_id", "meta_last_updated", "load_date", "row_hash")
          .orderBy("meta_last_updated"))